In [1]:
import pandas as pd
df_exp=pd.read_csv("C:/나이스/summary_experience_dataset.csv")
df_new=pd.read_csv("C:/나이스/summary_new_dataset.csv")

df_combined = pd.concat([df_exp['question_text'], df_new['question_text']], ignore_index=True)

# 결과를 새로운 DataFrame으로 만들기
df_questions = pd.DataFrame({'question_text': df_combined})
# 결과 확인
print(df_questions.head())
df_questions.to_csv('questions.csv')

                                       question_text
0                        디자이너로서 앞으로의 목표에 관해서 설명해 주세요
1  협업을 할 때 사교성이 좋은 편인지 궁금합니다 그리고 또 사교성을 키우기 위해 어떤...
2  지원자님이 태어나서 지금까지 한 일들 가운데 가장 후회했던 일이 무엇인가요 한 가지...
3  대학 생활을 보내면서 가장 힘들었던 경험은 무엇인가요 그것을 어떻게 극복하였는지 예...
4                                  가장 자신있는 작업은 무엇일까요


In [2]:
%pip uninstall JPype1 konlpy

^C
Note: you may need to restart the kernel to use updated packages.


In [3]:
# 질문별 빈도수 계산
question_counts = df_questions['question_text'].value_counts()

# 상위 5개 가장 많이 나온 질문 출력
top_questions = question_counts.head(5)

print("가장 많이 나온 질문 Top 5:")
print(top_questions)


가장 많이 나온 질문 Top 5:
question_text
회사에 입사를 했더니 업무 내용이나 강도가 예상했던 것과 많이 다르다면 어떻게 하시겠습니까                       19
커뮤니케이션을 잘 할 수 있는 지원자님만의 스킬이 있다면 한번 소개해 주실 수 있을까요                         17
회사의 덕목이 무엇이라고 생각하십니까 어떤 회사가 좋은 회사라고 생각하시는지 말씀해 주세요                       16
지금까지 협업을 하면서 가장 어려웠던 것과 그 어려움을 어떻게 해결했는지 함께 말씀해 주시기 바랍니다                 16
전공 외에 본인이 수준급의 실력이 될 정도로 노력한 것이 있다면 그것이 전공과 관련해서 어떤 도움을 주고 있는지도 궁금합니다    15
Name: count, dtype: int64


In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from konlpy.tag import Okt
import re
import random
import numpy as np

# KoNLPy Okt 형태소 분석기 초기화
okt = Okt()

# 텍스트 전처리 함수 (한국어 특화)
def clean_text(text):
    text = re.sub(r"[^가-힣\s]", "", text) # 한글과 공백만 남기고 제거
    return text.strip()

# 텍스트를 형태소 분석하여 명사만 추출하는 함수
def tokenize_korean_text(text):
    return ' '.join(okt.nouns(clean_text(text)))

def group_and_extract_questions(csv_file_path, num_clusters=5):
    """
    CSV 파일에서 질문을 로드하여 유사한 질문끼리 그룹화하고,
    각 그룹에서 랜덤으로 1개씩 질문을 추출합니다.

    Args:
        csv_file_path (str): 질문이 포함된 CSV 파일의 경로.
        num_clusters (int): 질문을 그룹화할 클러스터(그룹)의 개수.

    Returns:
        dict: 각 그룹에서 추출된 질문을 담은 딕셔너리.
              예: {'cluster_0': '질문 내용', 'cluster_1': '다른 질문 내용', ...}
    """
    try:
        df = df_questions
    except FileNotFoundError:
        print(f"오류: '{csv_file_path}' 파일을 찾을 수 없습니다.")
        return {}
    except Exception as e:
        print(f"CSV 파일 읽기 중 오류 발생: {e}")
        return {}

    if 'question_text' not in df.columns:
        print("오류: CSV 파일에 'question_text' 컬럼이 없습니다.")
        return {}

    # 텍스트 전처리 및 토큰화
    df['processed_question'] = df['question_text'].apply(tokenize_korean_text)

    # TF-IDF 벡터화
    # min_df를 조절하여 너무 자주 나오거나 너무 드물게 나오는 단어 필터링 가능
    vectorizer = TfidfVectorizer(max_features=1000, min_df=5, max_df=0.8)
    X = vectorizer.fit_transform(df['processed_question'])

    # K-Means 군집화
    # n_init='auto' 또는 명시적으로 횟수 지정 (예: 10)
    # n_init: KMeans 초기화 시도 횟수. 'auto'는 scikit-learn 버전 1.4부터 기본값이며, 이전 버전에서는 10이 일반적입니다.
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    df['cluster'] = kmeans.fit_predict(X)

    extracted_questions = {}
    for cluster_id in range(num_clusters):
        cluster_questions = df[df['cluster'] == cluster_id]['question_text'].tolist()
        if cluster_questions:
            extracted_questions[f'Cluster_{cluster_id}'] = random.choice(cluster_questions)
        else:
            extracted_questions[f'Cluster_{cluster_id}'] = "해당 클러스터에 질문이 없습니다."
            
    return extracted_questions

if __name__ == "__main__":
    
    
    # 원하는 클러스터(그룹) 개수를 num_clusters로 설정하세요.
    # 이 값에 따라 질문이 몇 개의 그룹으로 나뉠지 결정됩니다.
    # 너무 작으면 광범위하고, 너무 크면 너무 세분화될 수 있습니다.
    num_clusters_to_extract = 7 # 예시: 10개의 그룹으로 나누기

    selected_questions = group_and_extract_questions(df_questions, num_clusters=num_clusters_to_extract)

    print(f"그룹별로 추출된 질문 ({num_clusters_to_extract}개 그룹):")
    for group_name, question in selected_questions.items():
        print(f"- {group_name}: {question}")

    

그룹별로 추출된 질문 (7개 그룹):
- Cluster_0: 귀하께서는 개인적으로 경영 지식을 쌓기 위해 따로 공부하거나 노력한 점이 있다면 그것에 대해서 이야기해 주세요
- Cluster_1: 지원자님은 본인의 전공과 관련해서 어떤 부족함을 느껴서 따로 공부를 하고 계신 분야가 있으십니까
- Cluster_2: 본인만의 스트레스 해소법이 있는지 있다면 왜 그런 방식으로 스트레스를 해소하는지 말씀해 주실 수 있나요
- Cluster_3: 본인이 그동안 인생에서 겪었던 갈등들 중 가장 심각했던 갈등 한 가지만 골라 말씀해 주세요
- Cluster_4: 이제까지 살면서 전혀 몰랐던 환경에서 새로운 일을 맡아서 해 본 경험이 있다면 그 경험에 대해서 말씀해 주시겠어요
- Cluster_5: 본인이 직접 해외 출장 장소를 선택할 수 있다면 꼭 한 번 가서 일 해보고 싶은 나라는 어디인가요 그 이유는 무엇인가요
- Cluster_6: 만약 본인이 지금 지원한 이 직무가 아니라 다른 부서에 배치되게 될 경우 어떻게 하실 건가요 그래도 괜찮으실까요


In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from konlpy.tag import Okt
import re
import random
import numpy as np

# LangChain 및 LLM 관련 임포트
from langchain_core.prompts import PromptTemplate
from langchain_community.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from dotenv import load_dotenv
import os

# 환경 변수 로드 (OPENAI_API_KEY 등)
load_dotenv()

# KoNLPy Okt 형태소 분석기 초기화
okt = Okt()

# 텍스트 전처리 함수 (한국어 특화)
def clean_text(text):
    """
    텍스트에서 한글과 공백만 남기고 제거합니다.
    Args:
        text (str): 원본 텍스트.
    Returns:
        str: 정제된 텍스트.
    """
    text = re.sub(r"[^가-힣\s]", "", text) # 한글과 공백만 남기고 제거
    return text.strip()

# 텍스트를 형태소 분석하여 명사만 추출하는 함수
def tokenize_korean_text(text):
    """
    한국어 텍스트에서 명사만 추출하여 공백으로 구분된 문자열로 반환합니다.
    Args:
        text (str): 원본 텍스트.
    Returns:
        str: 명사만 추출된 텍스트.
    """
    return ' '.join(okt.nouns(clean_text(text)))

def get_clustered_dataframe(df_source, num_clusters=7):
    """
    원본 DataFrame을 받아 질문을 군집화하고, 'cluster' 컬럼이 추가된 DataFrame을 반환합니다.
    이 함수는 사용자가 제공한 'group_and_extract_questions' 함수의 핵심 군집화 로직을
    DataFrame을 직접 처리하도록 수정한 것입니다.
    
    Args:
        df_source (pd.DataFrame): 'question_text' 컬럼이 포함된 원본 DataFrame.
        num_clusters (int): 질문을 그룹화할 클러스터(그룹)의 개수.

    Returns:
        pd.DataFrame: 'processed_question' 및 'cluster' 컬럼이 추가된 DataFrame.
    """
    if df_source.empty or 'question_text' not in df_source.columns:
        print("오류: 입력 DataFrame이 비어있거나 'question_text' 컬럼이 없습니다.")
        return pd.DataFrame()

    df_copy = df_source.copy() # 원본 DataFrame을 변경하지 않도록 복사본 사용

    # 텍스트 전처리 및 토큰화
    df_copy['processed_question'] = df_copy['question_text'].apply(tokenize_korean_text)

    # TF-IDF 벡터화
    vectorizer = TfidfVectorizer(max_features=2000, min_df=5, max_df=0.8)
    X = vectorizer.fit_transform(df_copy['processed_question'])

    # K-Means 군집화
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    df_copy['cluster'] = kmeans.fit_predict(X)

    return df_copy


def generate_new_questions_for_cluster(cluster_id, clustered_df, llm_model, num_examples=5, num_to_generate=3):
    """
    특정 클러스터의 특징을 기반으로 새로운 질문을 생성합니다.

    Args:
        cluster_id (int): 질문을 생성할 클러스터의 ID.
        clustered_df (pd.DataFrame): 'cluster' 컬럼이 포함된 군집화된 DataFrame.
        llm_model: LangChain ChatOpenAI 객체.
        num_examples (int): 프롬프트에 포함할 클러스터 내 예시 질문의 개수.
        num_to_generate (int): 생성할 새로운 질문의 개수.

    Returns:
        list: 생성된 새로운 질문 리스트.
    """
    cluster_questions = clustered_df[clustered_df['cluster'] == cluster_id]['question_text'].tolist()

    if not cluster_questions:
        print(f"클러스터 {cluster_id}에 질문이 없습니다. 질문 생성을 건너뜝니다.")
        return []

    # 클러스터 내에서 무작위로 예시 질문 선택
    # 클러스터 내 질문 수가 num_examples보다 적으면 모든 질문 사용
    selected_examples = random.sample(cluster_questions, min(len(cluster_questions), num_examples))
    examples_str = "\n- ".join(selected_examples)

    # LLM 프롬프트 정의
    template = """
    당신은 면접 질문 생성 전문가입니다.
    아래는 특정 유형의 면접 질문 예시입니다. 이 예시들의 주제, 스타일, 질문의 의도 등을 고려하여,
    이와 유사한 새로운 면접 질문을 {num_to_generate}개 생성해 주세요.
    각 질문은 한 줄로 작성해 주시고, 번호를 붙여주세요.

    ---
    [면접 질문 예시]
    - {examples}
    ---

    [새로운 질문 생성]
    """
    prompt = PromptTemplate(input_variables=["examples", "num_to_generate"], template=template)
    chain = LLMChain(llm=llm_model, prompt=prompt)

    print(f"\n--- 클러스터 {cluster_id}에 대한 새로운 질문 생성 중 ---")
    print(f"  (예시 질문: {selected_examples[:3]}...)") # 일부만 출력

    try:
        response = chain.invoke({
            "examples": examples_str,
            "num_to_generate": num_to_generate
        })

        if isinstance(response, dict) and "text" in response:
            generated_text = response["text"].strip()
        elif hasattr(response, "content"): # for newer langchain versions
            generated_text = response.content.strip()
        else:
            generated_text = str(response).strip() # Fallback

        # 생성된 텍스트를 줄바꿈 기준으로 리스트로 분리
        new_questions = [q.strip() for q in generated_text.split('\n') if q.strip()]
        return new_questions

    except Exception as e:
        print(f"LLM 질문 생성 중 오류 발생: {e}")
        return []

if __name__ == "__main__":
    # --- 1. CSV 파일에서 DataFrame 로드 (이 부분은 사용자가 이미 가지고 있다고 가정) ---
    csv_file_path = 'questions.csv' # 실제 CSV 파일 경로로 변경하세요.
    try:
        df_questions = pd.read_csv(csv_file_path)
        if 'question_text' not in df_questions.columns:
            print("오류: CSV 파일에 'question_text' 컬럼이 없습니다. 스크립트를 종료합니다.")
            exit()
        print(f"'{csv_file_path}' 파일에서 {len(df_questions)}개의 질문을 로드했습니다.")
    except FileNotFoundError:
        print(f"오류: '{csv_file_path}' 파일을 찾을 수 없습니다. 스크립트를 종료합니다.")
        exit()
    except Exception as e:
        print(f"CSV 파일 로드 중 오류 발생: {e}. 스크립트를 종료합니다.")
        exit()

    # --- 2. DataFrame을 사용하여 질문 군집화 ---
    num_clusters_to_use = 7 # 군집화 및 생성에 사용할 클러스터 개수 (추천: 5~10)
    print(f"\n질문을 {num_clusters_to_use}개 클러스터로 군집화 중...")
    clustered_data_df = get_clustered_dataframe(df_questions, num_clusters=num_clusters_to_use)

    if not clustered_data_df.empty:
        # --- 3. LangChain LLM 초기화 및 각 클러스터에서 새로운 질문 생성 ---
        if os.getenv("OPENAI_API_KEY") is None:
            print("\n오류: OPENAI_API_KEY 환경 변수가 설정되지 않았습니다. .env 파일을 확인하거나 직접 설정해주세요. 질문 생성은 건너뜜니다.")
        else:
            llm = ChatOpenAI(model="gpt-4o", temperature=0.7, max_tokens=500) # 질문 생성에 적합한 모델과 설정

            all_generated_questions = {}
            for cluster_id in range(num_clusters_to_use):
                generated_q_list = generate_new_questions_for_cluster(
                    cluster_id=cluster_id,
                    clustered_df=clustered_data_df, # 군집화된 DataFrame 전달
                    llm_model=llm,
                    num_examples=5, # 각 클러스터에서 5개의 예시를 LLM에 전달
                    num_to_generate=3 # 각 클러스터당 3개의 새로운 질문 생성 시도
                )
                if generated_q_list:
                    all_generated_questions[f'Cluster_{cluster_id}'] = generated_q_list

            print("\n\n=== 모든 클러스터에 대한 생성된 질문 ===")
            if all_generated_questions:
                for cluster_name, questions in all_generated_questions.items():
                    print(f"\n[{cluster_name}]")
                    for q in questions:
                        print(f"- {q}")
            else:
                print("생성된 질문이 없습니다.")
    else:
        print("군집화할 데이터가 없거나 오류가 발생하여 추가 작업을 수행할 수 없습니다.")

'questions.csv' 파일에서 68074개의 질문을 로드했습니다.

질문을 7개 클러스터로 군집화 중...


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_17732\406196382.py:170: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model="gpt-4o", temperature=0.7, max_tokens=500) # 질문 생성에 적합한 모델과 설정
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_17732\406196382.py:118: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm_model, prompt=prompt)



--- 클러스터 0에 대한 새로운 질문 생성 중 ---
  (예시 질문: ['만약 지원자님이 하고 싶은 직무 말고 다른 직무를 맡게 되신다면 어떻게 대응하시겠습니까 그래도 괜찮으신지 궁금합니다', '직무 수행을 하다가 본인이 모르는 과제가 등장했다면 어떻게 해결하실 건가요 그 해결법과 이유를 말해 주세요', '직무 수행 시 난관에 봉착했다면 그 난관을 어떻게 해결하시겠습니까']...)

--- 클러스터 1에 대한 새로운 질문 생성 중 ---
  (예시 질문: ['최근 본인에게 가장 흥미로운 일은 무엇인가요 그리고 그 일이 왜 흥미로운지도 같이 설명해 주세요', '면접자께서 본인의 위치보다 어려운 목표를 달성한 경험을 설명 부탁드리겠습니다', '디자인 직무 수행에 있어서 재능의 역할은 얼마나 중요하다고 생각을 하시는지 여쭤보고 싶고요 지원자분께서는 재능 있는 디자이너 신 실지 한번 여쭤보고 싶습니다']...)

--- 클러스터 2에 대한 새로운 질문 생성 중 ---
  (예시 질문: ['팀으로 일하는 것과 혼자 일하는 것 중 무엇을 더욱 선호하시나요 그렇게 생각하시는 이유도 함께 들어볼 수 있을까요', '만약에 이직을 생각하는 동료를 만나게 된다고 가정을 해 봅시다 그렇다면 뭐라고 말씀을 하시겠어요 왜 그렇게 말씀하실 건지도 설명해 주세요', '만약 이직을 생각하는 동료를 만나게 된다면 지원자님은 뭐라고 말씀하실 건가요 또한 왜 그렇게 말씀하실 건지도 설명해 주시길 바랍니다']...)

--- 클러스터 3에 대한 새로운 질문 생성 중 ---
  (예시 질문: ['혹시 매일 공부하고 있는 분야가 따로 있으신가요 있다면 그걸 왜 공부하는지도 함께 설명해 주시기 바랍니다', '개인적으로 공부는 어떻게 하시나요 강의나 책을 활용한 공부 방법 같은 것을 말씀해 주시는 것도 좋습니다', '만약 본인에게 사내 스터디를 만들어서 공부할 수 있다고 한다면 어떤 것을 공부하고 싶으신가요 이유도 함께 말씀해 주시기 바랍니다']...)

--- 클러스터 4에 대한 새로운 질문 생성